In [ ]:
# -*- coding: utf-8 -*-
"""
RF RESÍDUO V11 PEAK-ALIGNED CENTERED LESS-REF — COMPENSAÇÃO POR ASSINATURA PRESERVADA VS PARK — 30–40 kHz
================================================================================

Objetivo
--------
Melhorar o RF físico V4, que ainda ficava ruidoso e longe da referência.

A mudança forte desta V10 continua parando de tentar transformar diretamente a curva toda.
Agora a compensação é feita assim:

    1) Monta H_ref = curva saudável, falha == 0, em REF_TEMP.
    2) Para cada temperatura T, monta H_T = curva saudável, falha == 0, em T.
    3) Alinha H_T em H_ref usando apenas curvas saudáveis.
    4) Para cada amostra X_T, calcula a assinatura estrutural:

           assinatura = X_T_alinhada - H_T_alinhada

    5) Remove ruído pequeno dessa assinatura usando um limiar aprendido nas curvas saudáveis.
    6) Reconstrói a curva compensada com centralização robusta sem copiar picos da referência:

           X_comp = mistura_adaptativa(
               H_ref + assinatura_filtrada,
               X_alinhada + (H_ref - H_T_alinhada)_suave
           )

Assim:
    - Dano 0 tende a ficar perto da referência.
    - Dano 1 e Dano 2 continuam diferentes da referência.
    - A compensação não tenta copiar a referência em cima do dano.

Ainda existe um Random Forest, mas agora ele é só um "cleanup" pequeno,
treinado apenas no saudável, para corrigir o erro restante depois dessa
compensação física por assinatura. Ele não tem liberdade para deformar picos.

Compara diretamente:
    Original × Park × RF resíduo V11

Saídas:
    - metricas_amostra_a_amostra.csv
    - resumo_metricas.csv
    - separacao_danos.csv
    - curvas exemplo
    - RMSD/CCDM por temperatura
    - comparação geral RMSD/CCDM

Como usar:
    Coloque este arquivo na mesma pasta de base-completo--.pkl e rode.
"""

# ============================================================
# 1) IMPORTS
# ============================================================

import os
import re
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

np.random.seed(42)


# ============================================================
# 2) CONFIGURAÇÕES PRINCIPAIS
# ============================================================

ARQ_BASE = "base-completo--.pkl"
REF_TEMP = 30.0

PASTA_SAIDA = "resultados_rf_residuo_v11_peakaligned_centered_lessref_30_40_vs_park"
PASTA_GRAFICOS = os.path.join(PASTA_SAIDA, "graficos")
PASTA_CURVAS = os.path.join(PASTA_GRAFICOS, "curvas_exemplo")
PASTA_METRICAS = os.path.join(PASTA_GRAFICOS, "metricas_por_temperatura")

for _p in [PASTA_SAIDA, PASTA_GRAFICOS, PASTA_CURVAS, PASTA_METRICAS]:
    os.makedirs(_p, exist_ok=True)

FAIXAS_ANALISE = [
    (30.0, 40.0),
    # Teste depois a região dos picos, se quiser:
    # (38.7, 41.7),
]

TEMPERATURAS_CURVAS = [-10, 30, 80]
TEMPERATURAS_ALVO_PLOT = list(range(-10, 81, 10))
DANOS = [0, 1, 2]
SALVAR_PDF = True

RUN_ORIGINAL = True
RUN_PARK = True
RUN_RF_RESIDUO_V11 = True


# ============================================================
# 3) PARK
# ============================================================

PARK_MAX_SHIFT_FRAC = 0.10
PARK_NSTEPS = 151
PARK_SMOOTH_WIN = 1


# ============================================================
# 4) RF RESÍDUO V10 CENTERED ORIGINAL-SAFE — PARÂMETROS IMPORTANTES
# ============================================================

# ------------------------------------------------------------
# 4.1 Alinhamento saudável H_T -> H_ref
# ------------------------------------------------------------
USE_HEALTHY_SHIFT = True
SHIFT_MAX_FRAC = 0.075
SHIFT_NSTEPS = 201
SHIFT_SMOOTH_WIN_FRAC = 0.055

# Em vez de ganho/offset agressivo na curva inteira, usamos apenas:
# - shift horizontal saudável;
# - offset robusto saudável;
# - tendência térmica suave saudável.
THERMAL_TREND_WIN_FRAC = 0.090
OFFSET_USE_MEDIAN = True

# ------------------------------------------------------------
# 4.2 Extração/preservação da assinatura de dano
# ------------------------------------------------------------
# Ganho da assinatura preservada.
# 1.00 preserva a diferença X_T - H_T.
# 0.85 deixa a curva mais perto da referência, mas pode reduzir dano.
# 1.10 amplifica um pouco o dano.
DAMAGE_SIGNATURE_GAIN = 0.72

# Limiar de ruído aprendido no saudável.
# Aumente se D0 ainda ficar ruidoso.
HEALTHY_NOISE_GAIN = 1.62
HEALTHY_NOISE_GLOBAL_FRAC = 0.09

# Denoise da assinatura: forte em regiões planas, fraco em picos.
SIGNATURE_DENOISE_WIN_FRAC = 0.016
SIGNATURE_DENOISE_STRENGTH = 0.56
SIGNATURE_PEAK_KEEP_GAIN = 8.5
SIGNATURE_PEAK_KEEP_MIN = 0.42

# Limite de assinatura para evitar explosões em borda/interpolação.
# É baseado na amplitude da referência. Se ainda aparecer pico absurdo na borda, reduza.
SIGNATURE_CLIP_FRAC_REF_AMP = 1.28

# Correção das bordas, onde interpolação pode criar artefato.
EDGE_TAPER_FRAC = 0.012

# ------------------------------------------------------------
# 4.3 RF cleanup pequeno, opcional
# ------------------------------------------------------------
USE_RF_CLEANUP = False
RF_CLEANUP_PARAMS = dict(
    n_estimators=350,
    max_depth=5,
    min_samples_leaf=4,
    min_samples_split=8,
    max_features="sqrt",
    bootstrap=True,
    n_jobs=-1,
    random_state=42,
)

N_ANCHORS_CLEANUP = 15
CLEANUP_TARGET_SMOOTH_WIN_FRAC = 0.100
CLEANUP_PRED_SMOOTH_WIN_FRAC = 0.100
CLEANUP_BLEND = 0.025
CLEANUP_CLAMP_GAIN = 0.45
CLEANUP_PEAK_PROTECT_GAIN = 12.0
CLEANUP_PEAK_PROTECT_MIN_WEIGHT = 0.10
CLEANUP_PEAK_PROTECT_SMOOTH_WIN_FRAC = 0.012

# ------------------------------------------------------------
# 4.4 Denoise final da curva compensada
# ------------------------------------------------------------
APPLY_FINAL_DENOISE = True
FINAL_DENOISE_WIN_FRAC = 0.016
FINAL_DENOISE_STRENGTH = 0.22
FINAL_DENOISE_PEAK_KEEP_GAIN = 7.0
FINAL_DENOISE_PEAK_KEEP_MIN = 0.55

# Métrica de assinatura local.
HF_SIGNATURE_SMOOTH_WIN_FRAC = 0.050

# ------------------------------------------------------------
# 4.X Reconstrução V10: baseada na curva original, não em H_ref
# ------------------------------------------------------------
# A V8 reconstruía como H_ref + assinatura_filtrada. Isso deixava D1/D2
# com cara de referência quando a assinatura era filtrada demais.
# A V10 usa a curva original alinhada como base e soma apenas a correção
# térmica saudável suave.
THERMAL_DELTA_WIN_FRAC = 0.125
THERMAL_DELTA_GAIN = 1.00

# Quanto usar da reconstrução antiga baseada em H_ref.
# Para saudável, pode usar bastante. Para dano, quase nada.
REF_BASE_BLEND_HEALTHY = 0.22
REF_BASE_BLEND_DAMAGE = 0.00
REF_BASE_BLEND_POWER = 2.25

# Reinjeta pouco detalhe local original, só quando a assinatura parece estrutural.
RAW_DETAIL_REINJECT_MAX = 0.08
RAW_DETAIL_REINJECT_WIN_FRAC = 0.020

# ------------------------------------------------------------
# 4.X.1 Centralização robusta sem copiar picos
# ------------------------------------------------------------
# Este bloco corrige o problema visual que apareceu no V9:
# - a curva precisava ficar mais centrada na referência;
# - mas aumentar a mistura com H_ref copiava detalhes/picos da referência.
#
# A solução aqui é uma centralização de baixa frequência:
# ajusta somente ganho pequeno + offset + inclinação em curva suavizada,
# com pesos baixos nos picos. Assim a linha de base aproxima da referência,
# mas os picos/vales estreitos não são transferidos da referência para o dano.
CENTER_BASELINE_ENABLE = True
CENTER_BASELINE_WIN_FRAC = 0.145
CENTER_ALPHA_HEALTHY = 0.78
CENTER_ALPHA_DAMAGE = 0.34
CENTER_ALPHA_POWER = 1.15
CENTER_GAIN_MIN = 0.94
CENTER_GAIN_MAX = 1.06
CENTER_OFFSET_CLIP_FRAC_REF_AMP = 0.26
CENTER_TILT_CLIP_FRAC_REF_AMP = 0.18
CENTER_PEAK_DOWNWEIGHT_GAIN = 7.5
CENTER_MIN_WEIGHT = 0.08
CENTER_CORRECTION_SMOOTH_FRAC = 0.210


# ------------------------------------------------------------
# 4.X.2 Pós-alinhamento de picos e centralização final
# ------------------------------------------------------------
# Corrige o que ainda sobrava no V10:
#   - picos um pouco deslocados em relação à referência;
#   - curva um pouco acima/abaixo da referência;
#   - sem deixar a forma final virar cópia da referência.
POST_ALIGN_ENABLE = True
POST_ALIGN_SHIFT_MAX_FRAC = 0.018
POST_ALIGN_SHIFT_NSTEPS = 81
POST_ALIGN_SMOOTH_WIN_FRAC = 0.030
POST_ALIGN_PEAK_WEIGHT_GAIN = 3.8
POST_ALIGN_STRUCTURAL_SHIFT_MIN = 0.22
POST_ALIGN_STRUCTURAL_SHIFT_MAX = 0.82
POST_ALIGN_FINAL_CENTER_ALPHA_HEALTHY = 0.62
POST_ALIGN_FINAL_CENTER_ALPHA_DAMAGE = 0.28
POST_ALIGN_GAIN_MIN = 0.965
POST_ALIGN_GAIN_MAX = 1.035
POST_ALIGN_OFFSET_CLIP_FRAC_REF_AMP = 0.12
POST_ALIGN_TILT_CLIP_FRAC_REF_AMP = 0.08
POST_ALIGN_CORRECTION_SMOOTH_FRAC = 0.150

# ------------------------------------------------------------
# 4.5 Ajustes V6 focados no Dano 1
# ------------------------------------------------------------
# O problema do V5 era tratar parte da assinatura suave do Dano 1 como ruído.
# Agora o código calcula um score de assinatura sem usar o rótulo do dano.
# Se a assinatura parece saudável, a filtragem fica forte.
# Se a assinatura parece dano, a filtragem preserva mais forma local/suave.
DAMAGE_SCORE_LOW = 0.78
DAMAGE_SCORE_HIGH = 2.35

# Separação da assinatura em tendência suave + detalhe local.
SIGNATURE_LOW_WIN_FRAC = 0.110
LOW_SIGNATURE_KEEP_HEALTHY = 0.02
LOW_SIGNATURE_KEEP_DAMAGE = 0.82

# Threshold adaptativo: para dano real, reduzimos o corte para não apagar D1.
THRESHOLD_DAMAGE_RELIEF = 0.78
SMOOTH_DAMAGE_RELIEF = 0.82

# Ganho adaptativo da assinatura: D1 estava forte/ruidoso demais em CCDM.
# Então o ganho fica menor para assinatura moderada e maior apenas quando a assinatura é muito forte.
ADAPTIVE_GAIN_HEALTHY = 0.18
ADAPTIVE_GAIN_DAMAGE = 1.16

# RF cleanup treinado no saudável pode deformar dano; por isso ele é reduzido
# quando a assinatura parece estrutural.
CLEANUP_DAMAGE_REDUCTION = 1.00

# ------------------------------------------------------------
# 4.6 Resgate do Dano 1 em temperaturas longe da referência
# ------------------------------------------------------------
# O Dano 1 em -10/80°C pode virar uma assinatura moderada: grande demais
# para ser saudável, mas pequena demais para sobreviver ao filtro forte.
# Estes parâmetros aumentam a preservação somente quando:
#   (a) a temperatura está longe da referência;
#   (b) a assinatura é maior que o ruído saudável.
# Não usa o rótulo falha.
TEMP_RESCUE_START_DELTA = 18.0
TEMP_RESCUE_FULL_DELTA = 50.0
TEMP_RESCUE_SCORE_LOW = 0.58
TEMP_RESCUE_SCORE_HIGH = 1.45
TEMP_RESCUE_MAX_EXTRA_WEIGHT = 0.68

# Reduz o limiar e preserva mais a componente suave quando longe da referência.
THRESHOLD_TEMP_EXTRA_RELIEF = 0.38
LOW_SIGNATURE_KEEP_FAR_EXTRA = 0.32
ADAPTIVE_GAIN_FAR_EXTRA = 0.28

# Mistura uma assinatura menos filtrada para evitar apagar D1 em -10/80°C.
TEMP_RESCUE_BLEND_MAX = 0.24
LOW_SIGNATURE_KEEP_MILD_BASE = 0.74
LOW_SIGNATURE_KEEP_MILD_EXTRA = 0.16

# Garante que a assinatura filtrada não fique quase zero quando ela parecia dano.
MIN_SIGNATURE_RMS_KEEP_NEAR = 0.28
MIN_SIGNATURE_RMS_KEEP_FAR = 0.55
MAX_SIGNATURE_RESCUE_SCALE = 1.80

# Resgate extra de formato local para dano moderado longe da referência.
# Ajuda especialmente o Dano 1 em 80°C, quando a assinatura é suave/moderada
# e o filtro anterior acabava deixando a curva quase igual à referência.
SHAPE_RESCUE_BLEND_MAX = 0.12
SHAPE_RESCUE_LOW_KEEP = 0.55
SHAPE_RESCUE_DETAIL_KEEP = 0.55

# Evita que o Dano 0 seja contaminado por resgate: só ativa se a assinatura
# passar claramente do ruído saudável por percentil alto.
QUANTILE_SCORE_LOW = 1.35
QUANTILE_SCORE_HIGH = 3.20


# ============================================================
# 5) ESTILO VISUAL
# ============================================================

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 18,
    "axes.labelsize": 22,
    "axes.titlesize": 24,
    "xtick.labelsize": 16,
    "ytick.labelsize": 17,
    "legend.fontsize": 14,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

CORES_DANO = {0: "tab:blue", 1: "tab:orange", 2: "tab:red"}
CORES_METODO = {
    "Original": "tab:red",
    "Referência": "black",
    "Park": "tab:green",
    "RF_residuo_v11_peakaligned_centered_lessref": "tab:blue",
}
NOMES_METODO = {
    "Original": "Original",
    "Park": "Park",
    "RF_residuo_v11_peakaligned_centered_lessref": "RF resíduo V11 peak-aligned centered less-ref",
}


# ============================================================
# 6) FUNÇÕES BÁSICAS
# ============================================================

def extract_freq_hz(col):
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None


def get_freq_columns(df, fmin_khz, fmax_khz):
    cols, freqs = [], []
    for c in df.columns:
        f = extract_freq_hz(c)
        if f is None:
            continue
        fk = f / 1e3
        if fmin_khz <= fk <= fmax_khz:
            cols.append(c)
            freqs.append(f)
    if len(cols) == 0:
        return [], np.array([])
    order = np.argsort(freqs)
    return [cols[i] for i in order], np.asarray(freqs, dtype=float)[order]


def make_odd_window(win, n, minimum=3):
    win = int(round(win))
    n = int(n)
    if n <= 3:
        return 1
    win = max(int(minimum), win)
    if win >= n:
        win = max(int(minimum), n // 5)
    if win % 2 == 0:
        win += 1
    if win >= n:
        win = n - 1
        if win % 2 == 0:
            win -= 1
    return max(1, int(win))


def odd_window_from_frac(n, frac, minimum=3):
    return make_odd_window(int(round(n * float(frac))), n, minimum=minimum)


def moving_average(arr, win):
    arr = np.asarray(arr, dtype=float)
    if len(arr) < 3 or win <= 1:
        return arr.copy()
    win = make_odd_window(win, len(arr))
    if win <= 1:
        return arr.copy()
    pad = win // 2
    arr_pad = np.pad(arr, (pad, pad), mode="edge")
    kernel = np.ones(win, dtype=float) / win
    y = np.convolve(arr_pad, kernel, mode="valid")
    if len(y) > len(arr):
        y = y[:len(arr)]
    elif len(y) < len(arr):
        y = np.pad(y, (0, len(arr) - len(y)), mode="edge")
    return y


def moving_average_matrix(X, win):
    X = np.asarray(X, dtype=float)
    return np.vstack([moving_average(row, win) for row in X])


def shift_interp(x, fHz, tau):
    f_shift = fHz + float(tau)
    return np.interp(fHz, f_shift, x, left=x[0], right=x[-1])


def pearson_corr(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    a0 = a - np.mean(a)
    b0 = b - np.mean(b)
    den = np.sqrt(np.sum(a0 ** 2) * np.sum(b0 ** 2)) + 1e-18
    return float(np.sum(a0 * b0) / den)


def rmsd(y, ref):
    y = np.asarray(y, dtype=float)
    ref = np.asarray(ref, dtype=float)
    return float(np.sqrt(np.mean((y - ref) ** 2)))


def ccdm(y, ref):
    return float(1.0 - pearson_corr(y, ref))


def faixa_label(fmin, fmax):
    if abs(fmin - round(fmin)) < 1e-9 and abs(fmax - round(fmax)) < 1e-9:
        return f"{int(round(fmin))}–{int(round(fmax))} kHz"
    return f"{fmin:.1f}–{fmax:.1f} kHz"


def format_temp(T):
    T = float(T)
    return str(int(T)) if T.is_integer() else f"{T:.1f}"


def estilo_eixos(ax):
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def salvar_fig(fig, pasta, nome_base):
    os.makedirs(pasta, exist_ok=True)
    png = os.path.join(pasta, nome_base + ".png")
    fig.savefig(png, dpi=600, bbox_inches="tight", facecolor="white")
    if SALVAR_PDF:
        pdf = os.path.join(pasta, nome_base + ".pdf")
        fig.savefig(pdf, bbox_inches="tight", facecolor="white")
        print(f"✅ Salvo:\n{png}\n{pdf}")
    else:
        print(f"✅ Salvo:\n{png}")


def escolher_temperaturas_validas(df, temps_alvo, danos=DANOS):
    temps_validas = []
    for T in sorted(df["temperatura_c"].dropna().unique()):
        ok = True
        for d in danos:
            if not np.any(np.isclose(df["temperatura_c"], T) & (df["falha"] == d)):
                ok = False
                break
        if ok:
            temps_validas.append(float(T))

    if len(temps_validas) == 0:
        raise ValueError("Nenhuma temperatura possui todos os danos.")

    temps_validas = np.asarray(temps_validas, dtype=float)
    rows, usadas = [], []
    for T_alvo in temps_alvo:
        idx = int(np.argmin(np.abs(temps_validas - float(T_alvo))))
        T_real = float(temps_validas[idx])
        rows.append({
            "temperatura_alvo_c": float(T_alvo),
            "temperatura_usada_c": T_real,
            "erro_abs_c": abs(T_real - float(T_alvo)),
        })
        if T_real not in usadas:
            usadas.append(T_real)
    return usadas, pd.DataFrame(rows)


# ============================================================
# 7) REFERÊNCIA SAUDÁVEL
# ============================================================

def get_healthy_references_by_temperature(df, fcols):
    df_sem = df[df["falha"] == 0].copy()
    if len(df_sem) == 0:
        raise ValueError("Não existem amostras sem dano para montar referência.")

    temps_h = np.array(sorted(df_sem["temperatura_c"].dropna().unique()), dtype=float)
    healthy_by_temp = {}
    for T in temps_h:
        pool = df_sem.loc[np.isclose(df_sem["temperatura_c"], T), fcols].to_numpy(float)
        healthy_by_temp[float(T)] = np.median(pool, axis=0)

    ref_temp_used = float(temps_h[np.argmin(np.abs(temps_h - REF_TEMP))])
    y_ref = healthy_by_temp[ref_temp_used].astype(float)

    print("✅ Referência montada SOMENTE com falha == 0")
    print(f"✅ Temperatura de referência saudável usada: {format_temp(ref_temp_used)}°C")
    if not np.isclose(ref_temp_used, REF_TEMP):
        print(f"⚠️ REF_TEMP={REF_TEMP}°C não existe exatamente. Usando {ref_temp_used}°C.")

    return healthy_by_temp, temps_h, y_ref, ref_temp_used


def interpolate_healthy_curve(healthy_by_temp, healthy_temps, T):
    healthy_temps = np.asarray(healthy_temps, dtype=float)
    T = float(T)
    if T <= healthy_temps[0]:
        return healthy_by_temp[float(healthy_temps[0])].copy(), float(healthy_temps[0])
    if T >= healthy_temps[-1]:
        return healthy_by_temp[float(healthy_temps[-1])].copy(), float(healthy_temps[-1])

    j = int(np.searchsorted(healthy_temps, T))
    T0, T1 = float(healthy_temps[j - 1]), float(healthy_temps[j])
    h0, h1 = healthy_by_temp[T0], healthy_by_temp[T1]
    w = (T - T0) / (T1 - T0 + 1e-12)
    return (1.0 - w) * h0 + w * h1, T


# ============================================================
# 8) ALINHAMENTO SAUDÁVEL E EXTRAÇÃO DE ASSINATURA
# ============================================================

def robust_weights_for_shift(x, ref):
    x = np.asarray(x, dtype=float)
    ref = np.asarray(ref, dtype=float)
    d2x = np.abs(np.gradient(np.gradient(x)))
    d2r = np.abs(np.gradient(np.gradient(ref)))
    score = d2x + d2r
    scale = np.percentile(score, 60) + 1e-12
    w = 1.0 / (1.0 + (score / scale) ** 1.25)
    return np.clip(w, 0.10, 1.0)


def estimate_healthy_shift(h_T, h_ref, fHz):
    if not USE_HEALTHY_SHIFT:
        return 0.0

    df_band = float(fHz[-1] - fHz[0])
    tau_max = SHIFT_MAX_FRAC * df_band
    taus = np.linspace(-tau_max, tau_max, SHIFT_NSTEPS)

    win = odd_window_from_frac(len(h_T), SHIFT_SMOOTH_WIN_FRAC, minimum=7)
    hT_s = moving_average(h_T, win)
    ref_s = moving_average(h_ref, win)

    best_tau = 0.0
    best_err = np.inf
    for tau in taus:
        h_shift = shift_interp(hT_s, fHz, tau)
        w = robust_weights_for_shift(h_shift, ref_s)
        offset = np.average(ref_s - h_shift, weights=w)
        err = np.average((h_shift + offset - ref_s) ** 2, weights=w)
        if err < best_err:
            best_err = err
            best_tau = tau
    return float(best_tau)


def peakness_score(x, win):
    x = np.asarray(x, dtype=float)
    if len(x) < 5:
        return np.zeros_like(x)
    xs = moving_average(x, win)
    d1 = np.gradient(xs)
    d2 = np.gradient(d1)
    p = np.abs(d2)
    scale = np.percentile(p, 85) + 1e-18
    return p / scale


def peak_keep_weight(x, win, gain=4.0, min_keep=0.18):
    p = peakness_score(x, win)
    keep = (gain * p) / (1.0 + gain * p)
    keep = np.clip(keep, min_keep, 1.0)
    return moving_average(keep, win)


def edge_taper(n, frac):
    k = int(round(n * frac))
    if k <= 1:
        return np.ones(n)
    w = np.ones(n)
    ramp = np.linspace(0.0, 1.0, k)
    w[:k] = ramp
    w[-k:] = ramp[::-1]
    return w


def soft_threshold(x, thr):
    x = np.asarray(x, dtype=float)
    thr = np.asarray(thr, dtype=float)
    return np.sign(x) * np.maximum(np.abs(x) - thr, 0.0)


def estimate_noise_floor_from_healthy(df, fcols, fHz, healthy_by_temp, healthy_temps, y_ref):
    """Aprende o ruído típico da assinatura X_T - H_T usando só falha == 0."""
    df_h = df[df["falha"] == 0].copy()
    Xh = df_h[fcols].to_numpy(float)
    Th = df_h["temperatura_c"].to_numpy(float)

    residuals = []
    info = []
    for x, T in zip(Xh, Th):
        h_T, T_used = interpolate_healthy_curve(healthy_by_temp, healthy_temps, T)
        tau = estimate_healthy_shift(h_T, y_ref, fHz)
        x_shift = shift_interp(x, fHz, tau)
        h_shift = shift_interp(h_T, fHz, tau)

        # Remove offset global para sobrar só ruído/assinatura espúria do saudável.
        r = x_shift - h_shift
        r = r - np.median(r)
        residuals.append(r)
        info.append({"T": T, "T_healthy_used": T_used, "tau_hz": tau})

    R = np.asarray(residuals, dtype=float)
    local = np.percentile(np.abs(R), 75, axis=0)
    global_noise = np.median(np.abs(R)) + 1e-12
    floor = HEALTHY_NOISE_GAIN * local + HEALTHY_NOISE_GLOBAL_FRAC * global_noise
    return floor.astype(float), pd.DataFrame(info)


def signature_damage_score(signature, noise_floor):
    """
    Score sem usar rótulo: quão maior a assinatura é que o ruído saudável.

    A V10 usa dois sinais:
    1) RMS global: pega assinatura larga/suave.
    2) Quantil alto de |assinatura|/ruído: pega D1 quando só alguns trechos mudam.

    O retorno é o maior dos dois. Isso deixa o D1 em temperatura distante
    aparecer mais, sem precisar usar falha == 1 dentro da compensação.
    """
    sig = np.asarray(signature, dtype=float)
    nf = np.asarray(noise_floor, dtype=float)
    sig0 = sig - np.median(sig)
    rms_sig = np.sqrt(np.mean(sig0 ** 2))
    rms_noise = np.sqrt(np.mean(nf ** 2)) + 1e-12
    score_rms = rms_sig / rms_noise

    ratio = np.abs(sig0) / (nf + 1e-12)
    score_q = np.percentile(ratio, 88) / 2.0
    return float(max(score_rms, score_q))


def quantile_structural_weight(signature, noise_floor):
    """Peso auxiliar para ativar resgate só quando há trechos acima do ruído saudável."""
    sig = np.asarray(signature, dtype=float)
    nf = np.asarray(noise_floor, dtype=float)
    sig0 = sig - np.median(sig)
    ratio = np.abs(sig0) / (nf + 1e-12)
    q = float(np.percentile(ratio, 88))
    u = (q - QUANTILE_SCORE_LOW) / (QUANTILE_SCORE_HIGH - QUANTILE_SCORE_LOW + 1e-12)
    return float(np.clip(u, 0.0, 1.0))


def damage_weight_from_score(score):
    """0 = parece saudável; 1 = parece assinatura estrutural forte."""
    u = (float(score) - DAMAGE_SCORE_LOW) / (DAMAGE_SCORE_HIGH - DAMAGE_SCORE_LOW + 1e-12)
    return float(np.clip(u, 0.0, 1.0))


def temp_distance_weight(T):
    """0 perto da referência; 1 bem longe da referência."""
    if T is None:
        return 0.0
    dT = abs(float(T) - float(REF_TEMP))
    u = (dT - TEMP_RESCUE_START_DELTA) / (TEMP_RESCUE_FULL_DELTA - TEMP_RESCUE_START_DELTA + 1e-12)
    return float(np.clip(u, 0.0, 1.0))


def soft_score_weight(score):
    """Peso para assinatura moderada: usado para resgatar D1 sem usar rótulo."""
    u = (float(score) - TEMP_RESCUE_SCORE_LOW) / (TEMP_RESCUE_SCORE_HIGH - TEMP_RESCUE_SCORE_LOW + 1e-12)
    return float(np.clip(u, 0.0, 1.0))


def denoise_signature(signature, noise_floor, ref_amp, T=None):
    """
    V10 centered original-safe.

    Esta é a alteração principal para o caso que você mostrou:
    em temperaturas longe da referência, o Dano 1 pode ser uma assinatura
    moderada. Se o filtro for forte demais, a curva vira quase a referência
    e o D1 desaparece.

    O filtro continua sem usar o rótulo do dano. Ele só olha:
      - intensidade da assinatura em relação ao ruído saudável;
      - distância de temperatura até REF_TEMP.
    """
    sig = np.asarray(signature, dtype=float)
    n = len(sig)

    score = signature_damage_score(sig, noise_floor)
    dw_score = damage_weight_from_score(score)
    tw = temp_distance_weight(T)
    sw = soft_score_weight(score)
    qw = quantile_structural_weight(sig, noise_floor)

    # Peso final de dano: score manda; temperatura longe só ajuda se a assinatura
    # já passou minimamente do ruído saudável.
    dw_extra = TEMP_RESCUE_MAX_EXTRA_WEIGHT * tw * max(sw, qw)
    dw = float(np.clip(max(dw_score, dw_extra), 0.0, 1.0))

    # Tendência suave: importante para D1. No V6/V5 parte dela podia ser apagada.
    win_low = odd_window_from_frac(n, SIGNATURE_LOW_WIN_FRAC, minimum=21)
    low = moving_average(sig, win_low)
    detail = sig - low

    low_centered = low - np.median(low)
    low_keep = LOW_SIGNATURE_KEEP_HEALTHY + (LOW_SIGNATURE_KEEP_DAMAGE - LOW_SIGNATURE_KEEP_HEALTHY) * dw
    low_keep += LOW_SIGNATURE_KEEP_FAR_EXTRA * tw * max(sw, qw)
    low_keep = float(np.clip(low_keep, 0.0, 0.90))
    low_part = low_keep * low_centered

    # Detalhes locais: picos/vales são mais preservados; regiões planas são suavizadas.
    win = odd_window_from_frac(n, SIGNATURE_DENOISE_WIN_FRAC, minimum=5)
    keep = peak_keep_weight(detail, win, gain=SIGNATURE_PEAK_KEEP_GAIN, min_keep=SIGNATURE_PEAK_KEEP_MIN)

    thr_scale = 1.0 - THRESHOLD_DAMAGE_RELIEF * dw - THRESHOLD_TEMP_EXTRA_RELIEF * tw * max(sw, qw)
    thr_scale = float(np.clip(thr_scale, 0.16, 1.0))
    thr = noise_floor * thr_scale * (1.0 - 0.76 * keep)
    detail_thr = soft_threshold(detail, thr)

    detail_smooth = moving_average(detail_thr, win)
    smooth_strength = SIGNATURE_DENOISE_STRENGTH * (1.0 - keep) * (1.0 - SMOOTH_DAMAGE_RELIEF * dw)
    smooth_strength *= (1.0 - 0.45 * tw * max(sw, qw))
    smooth_strength = np.clip(smooth_strength, 0.0, 1.0)
    detail_part = (1.0 - smooth_strength) * detail_thr + smooth_strength * detail_smooth

    gain = ADAPTIVE_GAIN_HEALTHY + (ADAPTIVE_GAIN_DAMAGE - ADAPTIVE_GAIN_HEALTHY) * dw
    gain *= (1.0 + ADAPTIVE_GAIN_FAR_EXTRA * tw * max(sw, qw))
    gain = float(np.clip(gain, 0.0, 1.25))

    sig_f = gain * (low_part + detail_part)

    # Assinatura alternativa menos filtrada, usada só como resgate parcial
    # em temperaturas distantes e assinatura moderada/forte.
    mild_low_keep = LOW_SIGNATURE_KEEP_MILD_BASE + LOW_SIGNATURE_KEEP_MILD_EXTRA * tw
    mild_low_keep = float(np.clip(mild_low_keep, 0.0, 0.90))
    low_mild = mild_low_keep * low_centered

    detail_mild_smooth = moving_average(detail, win)
    detail_mild = keep * detail + (1.0 - keep) * (0.70 * detail + 0.30 * detail_mild_smooth)
    sig_mild = low_mild + detail_mild

    rescue_w = TEMP_RESCUE_BLEND_MAX * tw * max(sw, qw) * (0.35 + 0.65 * dw)
    rescue_w = float(np.clip(rescue_w, 0.0, TEMP_RESCUE_BLEND_MAX))
    sig_f = (1.0 - rescue_w) * sig_f + rescue_w * sig_mild

    # Resgate extra de formato: preserva parte da assinatura crua quando a
    # temperatura está longe e existem trechos acima do ruído saudável.
    # Isso é exatamente o caso em que o D1 em 80°C parecia virar saudável.
    shape_w = SHAPE_RESCUE_BLEND_MAX * tw * max(sw, qw) * (0.20 + 0.80 * dw)
    shape_w = float(np.clip(shape_w, 0.0, SHAPE_RESCUE_BLEND_MAX))
    sig_shape = SHAPE_RESCUE_LOW_KEEP * low_centered + SHAPE_RESCUE_DETAIL_KEEP * detail
    sig_f = (1.0 - shape_w) * sig_f + shape_w * sig_shape

    # Proteção contra "apagar" D1: se a assinatura filtrada ficou pequena demais
    # em relação à assinatura original, reescala moderadamente.
    raw_rms = float(np.sqrt(np.mean((sig - np.median(sig)) ** 2)))
    filt_rms = float(np.sqrt(np.mean(sig_f ** 2))) + 1e-12
    min_keep = MIN_SIGNATURE_RMS_KEEP_NEAR + (MIN_SIGNATURE_RMS_KEEP_FAR - MIN_SIGNATURE_RMS_KEEP_NEAR) * tw
    min_keep *= max(sw, qw)
    scale_used = 1.0
    if raw_rms > 1e-12 and dw > 0.12:
        target_rms = min_keep * raw_rms
        if filt_rms < target_rms:
            scale_used = float(np.clip(target_rms / filt_rms, 1.0, MAX_SIGNATURE_RESCUE_SCALE))
            sig_f = sig_f * scale_used

    taper = edge_taper(n, EDGE_TAPER_FRAC)
    sig_f = sig_f * taper
    sig_f = np.clip(sig_f, -SIGNATURE_CLIP_FRAC_REF_AMP * ref_amp, SIGNATURE_CLIP_FRAC_REF_AMP * ref_amp)

    return sig_f.astype(float), {
        "score": score,
        "damage_weight": dw,
        "damage_weight_score": dw_score,
        "temp_weight": tw,
        "soft_score_weight": sw,
        "quantile_weight": qw,
        "shape_rescue_weight": shape_w,
        "gain": gain,
        "low_keep": low_keep,
        "rescue_weight": rescue_w,
        "min_keep": min_keep,
        "scale_used": scale_used,
        "raw_signature_rms": raw_rms,
        "filtered_signature_rms": float(np.sqrt(np.mean(sig_f ** 2))),
    }


def final_denoise_curve(y):
    y = np.asarray(y, dtype=float)
    if not APPLY_FINAL_DENOISE:
        return y.copy()

    n = len(y)
    win = odd_window_from_frac(n, FINAL_DENOISE_WIN_FRAC, minimum=5)
    ys = moving_average(y, win)
    keep = peak_keep_weight(y, win, gain=FINAL_DENOISE_PEAK_KEEP_GAIN, min_keep=FINAL_DENOISE_PEAK_KEEP_MIN)
    w_smooth = FINAL_DENOISE_STRENGTH * (1.0 - keep)
    return (1.0 - w_smooth) * y + w_smooth * ys


def reconstruction_ref_blend(sig_info):
    """
    Define quanto usar da reconstrução baseada na referência.

    Se parece saudável:
        usa mais H_ref, porque D0 deve colar na referência.

    Se parece dano:
        usa mais a curva original alinhada, para não apagar D1/D2.
    """
    dw = float(sig_info.get("damage_weight", 0.0))
    tw = float(sig_info.get("temp_weight", 0.0))
    sw = float(sig_info.get("soft_score_weight", 0.0))
    qw = float(sig_info.get("quantile_weight", 0.0))

    structural = max(dw, 0.65 * tw * max(sw, qw))
    structural = float(np.clip(structural, 0.0, 1.0))

    blend = REF_BASE_BLEND_DAMAGE + (REF_BASE_BLEND_HEALTHY - REF_BASE_BLEND_DAMAGE) * ((1.0 - structural) ** REF_BASE_BLEND_POWER)
    blend = float(np.clip(blend, REF_BASE_BLEND_DAMAGE, REF_BASE_BLEND_HEALTHY))
    return blend, structural



def robust_center_weights(y_s, ref_s):
    """Pesos para centralização: quase ignora picos estreitos."""
    win = odd_window_from_frac(len(y_s), 0.012, minimum=5)
    py = peakness_score(y_s, win)
    pr = peakness_score(ref_s, win)
    p = py + pr
    w = 1.0 / (1.0 + CENTER_PEAK_DOWNWEIGHT_GAIN * p)
    return np.clip(w, CENTER_MIN_WEIGHT, 1.0)


def center_curve_without_copying_ref_peaks(y, y_ref, sig_info):
    """
    Centraliza a curva na referência sem copiar detalhes da referência.

    O ajuste é feito em versões bem suavizadas de y e y_ref, com pesos baixos
    nos picos. Depois aplica só uma correção afim suave:

        y_center = y + alpha * [(a - 1) y + b + c z]

    Como a correção é de baixa frequência, ela corrige offset/inclinação/escala
    sem injetar os picos estreitos da referência dentro dos danos.
    """
    if not CENTER_BASELINE_ENABLE:
        return np.asarray(y, dtype=float)

    y = np.asarray(y, dtype=float)
    y_ref = np.asarray(y_ref, dtype=float)
    n = len(y)
    z = np.linspace(-1.0, 1.0, n)

    win = odd_window_from_frac(n, CENTER_BASELINE_WIN_FRAC, minimum=31)
    y_s = moving_average(y, win)
    ref_s = moving_average(y_ref, win)

    A = np.column_stack([y_s, np.ones(n), z])
    w = robust_center_weights(y_s, ref_s)
    sw = np.sqrt(w)

    try:
        coef, *_ = np.linalg.lstsq(A * sw[:, None], ref_s * sw, rcond=None)
        a, b, c = [float(v) for v in coef]
    except Exception:
        a, b, c = 1.0, float(np.median(ref_s - y_s)), 0.0

    ref_amp = np.ptp(y_ref) + 1e-12
    a = float(np.clip(a, CENTER_GAIN_MIN, CENTER_GAIN_MAX))
    b = float(np.clip(b, -CENTER_OFFSET_CLIP_FRAC_REF_AMP * ref_amp, CENTER_OFFSET_CLIP_FRAC_REF_AMP * ref_amp))
    c = float(np.clip(c, -CENTER_TILT_CLIP_FRAC_REF_AMP * ref_amp, CENTER_TILT_CLIP_FRAC_REF_AMP * ref_amp))

    # structural perto de 1 significa que a curva parece ter dano estrutural.
    _, structural = reconstruction_ref_blend(sig_info)
    alpha = CENTER_ALPHA_DAMAGE + (CENTER_ALPHA_HEALTHY - CENTER_ALPHA_DAMAGE) * ((1.0 - structural) ** CENTER_ALPHA_POWER)
    alpha = float(np.clip(alpha, CENTER_ALPHA_DAMAGE, CENTER_ALPHA_HEALTHY))

    correction = (a - 1.0) * y + b + c * z

    # deixa a correção ainda mais suave para não virar cópia da referência.
    win_corr = odd_window_from_frac(n, CENTER_CORRECTION_SMOOTH_FRAC, minimum=31)
    correction_s = moving_average(correction, win_corr)

    y_center = y + alpha * correction_s

    # salva diagnósticos dentro do sig_info para o CSV.
    sig_info["center_alpha"] = alpha
    sig_info["center_gain"] = a
    sig_info["center_offset"] = b
    sig_info["center_tilt"] = c

    return y_center.astype(float)


def post_align_curve_peaks_and_baseline(y, y_ref, fHz, sig_info):
    """
    Pós-ajuste leve para alinhar melhor os picos e recentrar a linha de base,
    sem puxar a curva para virar uma cópia da referência.

    Estratégia:
      1) estima um shift residual pequeno em versões suavizadas, com peso extra
         nos picos da própria curva e da referência;
      2) aplica só uma fração desse shift quando a curva parece estrutural;
      3) corrige ganho/offset/inclinação apenas em baixa frequência.
    """
    y = np.asarray(y, dtype=float)
    y_ref = np.asarray(y_ref, dtype=float)
    if (not POST_ALIGN_ENABLE) or len(y) < 5:
        return y

    n = len(y)
    z = np.linspace(-1.0, 1.0, n)
    _, structural = reconstruction_ref_blend(sig_info)

    # --- shift residual pequeno e peak-aware ---
    win = odd_window_from_frac(n, POST_ALIGN_SMOOTH_WIN_FRAC, minimum=7)
    ys = moving_average(y, win)
    rs = moving_average(y_ref, win)

    df_band = float(fHz[-1] - fHz[0]) if len(fHz) >= 2 else float(n - 1)
    tau_max = POST_ALIGN_SHIFT_MAX_FRAC * df_band
    taus = np.linspace(-tau_max, tau_max, POST_ALIGN_SHIFT_NSTEPS)

    py = peakness_score(ys, win)
    pr = peakness_score(rs, win)
    w_peak = 1.0 + POST_ALIGN_PEAK_WEIGHT_GAIN * (py + pr)
    w_peak = np.clip(w_peak, 0.25, None)

    best_tau = 0.0
    best_err = np.inf
    for tau in taus:
        y_try = shift_interp(ys, fHz, tau)
        offset = np.average(rs - y_try, weights=w_peak)
        err = np.average((y_try + offset - rs) ** 2, weights=w_peak)
        if err < best_err:
            best_err = err
            best_tau = float(tau)

    alpha_shift = POST_ALIGN_STRUCTURAL_SHIFT_MAX - (POST_ALIGN_STRUCTURAL_SHIFT_MAX - POST_ALIGN_STRUCTURAL_SHIFT_MIN) * structural
    alpha_shift = float(np.clip(alpha_shift, POST_ALIGN_STRUCTURAL_SHIFT_MIN, POST_ALIGN_STRUCTURAL_SHIFT_MAX))
    tau_apply = alpha_shift * best_tau
    y_shift = shift_interp(y, fHz, tau_apply)

    # --- centralização leve de baixa frequência ---
    win2 = odd_window_from_frac(n, max(CENTER_BASELINE_WIN_FRAC, POST_ALIGN_CORRECTION_SMOOTH_FRAC), minimum=31)
    y2 = moving_average(y_shift, win2)
    r2 = moving_average(y_ref, win2)
    w2 = robust_center_weights(y2, r2)
    A = np.column_stack([y2, np.ones(n), z])
    sw = np.sqrt(w2)
    try:
        coef, *_ = np.linalg.lstsq(A * sw[:, None], r2 * sw, rcond=None)
        a, b, c = [float(v) for v in coef]
    except Exception:
        a, b, c = 1.0, float(np.median(r2 - y2)), 0.0

    ref_amp = np.ptp(y_ref) + 1e-12
    a = float(np.clip(a, POST_ALIGN_GAIN_MIN, POST_ALIGN_GAIN_MAX))
    b = float(np.clip(b, -POST_ALIGN_OFFSET_CLIP_FRAC_REF_AMP * ref_amp, POST_ALIGN_OFFSET_CLIP_FRAC_REF_AMP * ref_amp))
    c = float(np.clip(c, -POST_ALIGN_TILT_CLIP_FRAC_REF_AMP * ref_amp, POST_ALIGN_TILT_CLIP_FRAC_REF_AMP * ref_amp))

    alpha_center = POST_ALIGN_FINAL_CENTER_ALPHA_DAMAGE + (POST_ALIGN_FINAL_CENTER_ALPHA_HEALTHY - POST_ALIGN_FINAL_CENTER_ALPHA_DAMAGE) * ((1.0 - structural) ** 1.05)
    alpha_center = float(np.clip(alpha_center, POST_ALIGN_FINAL_CENTER_ALPHA_DAMAGE, POST_ALIGN_FINAL_CENTER_ALPHA_HEALTHY))

    correction = (a - 1.0) * y_shift + b + c * z
    win_corr = odd_window_from_frac(n, POST_ALIGN_CORRECTION_SMOOTH_FRAC, minimum=21)
    correction_s = moving_average(correction, win_corr)
    y_out = y_shift + alpha_center * correction_s

    sig_info["post_tau_raw"] = best_tau
    sig_info["post_tau_apply"] = tau_apply
    sig_info["post_shift_alpha"] = alpha_shift
    sig_info["post_center_alpha"] = alpha_center
    sig_info["post_gain"] = a
    sig_info["post_offset"] = b
    sig_info["post_tilt"] = c

    return y_out.astype(float)


def reconstruct_original_based(x_shift, h_shift, y_ref, fHz, signature, sig_info):
    """
    Reconstrução nova da V10.

    Em vez de forçar:
        y = H_ref + assinatura_filtrada

    usa:
        y = X_alinhada + correção_térmica_saudável_suave

    Isso preserva melhor a forma própria do dano.
    """
    n = len(y_ref)

    # Correção térmica saudável: leva H_T alinhada para H_ref.
    thermal_delta = y_ref - h_shift

    # Suaviza para não copiar picos estreitos da referência para dentro do dano.
    win_delta = odd_window_from_frac(n, THERMAL_DELTA_WIN_FRAC, minimum=21)
    thermal_delta_s = moving_average(thermal_delta, win_delta)

    y_orig_based = x_shift + THERMAL_DELTA_GAIN * thermal_delta_s

    # Pequena reinjeção de detalhe local original quando parece dano.
    # Não usa rótulo; usa o score de assinatura calculado acima.
    _, structural = reconstruction_ref_blend(sig_info)

    win_detail = odd_window_from_frac(n, RAW_DETAIL_REINJECT_WIN_FRAC, minimum=5)
    sig_low = moving_average(signature, win_detail)
    sig_detail = signature - sig_low

    keep = peak_keep_weight(
        sig_detail,
        win_detail,
        gain=SIGNATURE_PEAK_KEEP_GAIN,
        min_keep=SIGNATURE_PEAK_KEEP_MIN,
    )

    reinject = RAW_DETAIL_REINJECT_MAX * structural
    y_orig_based = y_orig_based + reinject * keep * sig_detail

    # Centraliza linha de base sem puxar picos da referência.
    # Esta é a principal correção do V10 em relação ao V9.
    y_orig_based = center_curve_without_copying_ref_peaks(y_orig_based, y_ref, sig_info)

    return y_orig_based.astype(float)


def signature_compensation_curve(x, T, fHz, healthy_by_temp, healthy_temps, y_ref, noise_floor):
    """
    V10 centered original-safe.

    Diferença principal em relação ao V8:
    - V8 fazia: y_comp = H_ref + assinatura_filtrada
    - V10 faz: y_comp = X_alinhada + correção_térmica_saudável_suave

    Resultado esperado:
    - D0 continua perto da referência.
    - D1/D2 preservam mais características próprias.
    - Reduz a mistura visual entre curva de dano e curva de referência.
    """
    h_T, T_used = interpolate_healthy_curve(healthy_by_temp, healthy_temps, T)
    tau = estimate_healthy_shift(h_T, y_ref, fHz)

    x_shift = shift_interp(x, fHz, tau)
    h_shift = shift_interp(h_T, fHz, tau)

    # Assinatura estrutural antes do filtro.
    signature_raw = x_shift - h_shift

    # Remove offset global, mas mantém forma local/larga centrada.
    signature = signature_raw - np.median(signature_raw)

    ref_amp = np.ptp(y_ref) + 1e-12

    # Continua calculando assinatura filtrada, mas agora ela NÃO domina D1/D2.
    signature_f, sig_info = denoise_signature(signature, noise_floor, ref_amp, T=T)

    # Reconstrução antiga: excelente para D0, perigosa para dano.
    y_ref_based = y_ref + DAMAGE_SIGNATURE_GAIN * signature_f

    # Reconstrução nova: preserva melhor o formato da amostra original.
    y_orig_based = reconstruct_original_based(
        x_shift=x_shift,
        h_shift=h_shift,
        y_ref=y_ref,
        fHz=fHz,
        signature=signature,
        sig_info=sig_info,
    )

    # Mistura adaptativa sem usar falha:
    # saudável -> mais y_ref_based;
    # dano -> mais y_orig_based.
    ref_blend, structural = reconstruction_ref_blend(sig_info)
    y_comp = ref_blend * y_ref_based + (1.0 - ref_blend) * y_orig_based

    # Pós-alinhamento leve: melhora a coincidência de posição dos picos e
    # centraliza a linha de base, mas sem forçar o formato da referência.
    y_comp = post_align_curve_peaks_and_baseline(y_comp, y_ref, fHz, sig_info)
    y_comp = final_denoise_curve(y_comp)

    info = {
        "T": float(T),
        "T_healthy_used": float(T_used),
        "tau_hz": float(tau),
        "signature_rms": float(np.sqrt(np.mean(signature_f ** 2))),
        "signature_score": float(sig_info["score"]),
        "damage_weight": float(sig_info["damage_weight"]),
        "adaptive_gain": float(sig_info["gain"]),
        "low_keep": float(sig_info["low_keep"]),
        "temp_weight": float(sig_info.get("temp_weight", 0.0)),
        "soft_score_weight": float(sig_info.get("soft_score_weight", 0.0)),
        "quantile_weight": float(sig_info.get("quantile_weight", 0.0)),
        "rescue_weight": float(sig_info.get("rescue_weight", 0.0)),
        "shape_rescue_weight": float(sig_info.get("shape_rescue_weight", 0.0)),
        "min_keep": float(sig_info.get("min_keep", 0.0)),
        "scale_used": float(sig_info.get("scale_used", 1.0)),
        "raw_signature_rms": float(sig_info.get("raw_signature_rms", 0.0)),
        "filtered_signature_rms": float(sig_info.get("filtered_signature_rms", 0.0)),
        "ref_blend": float(ref_blend),
        "structural_weight": float(structural),
        "center_alpha": float(sig_info.get("center_alpha", np.nan)),
        "center_gain": float(sig_info.get("center_gain", np.nan)),
        "center_offset": float(sig_info.get("center_offset", np.nan)),
        "center_tilt": float(sig_info.get("center_tilt", np.nan)),
        "post_tau_raw": float(sig_info.get("post_tau_raw", np.nan)),
        "post_tau_apply": float(sig_info.get("post_tau_apply", np.nan)),
        "post_shift_alpha": float(sig_info.get("post_shift_alpha", np.nan)),
        "post_center_alpha": float(sig_info.get("post_center_alpha", np.nan)),
        "post_gain": float(sig_info.get("post_gain", np.nan)),
        "post_offset": float(sig_info.get("post_offset", np.nan)),
        "post_tilt": float(sig_info.get("post_tilt", np.nan)),
    }
    return y_comp.astype(float), info


# ============================================================
# 9) PARK
# ============================================================

def park_single(x, ref, fHz):
    df_band = float(fHz[-1] - fHz[0])
    tau_max = PARK_MAX_SHIFT_FRAC * df_band
    best_err = np.inf
    best_tau = 0.0
    best_dS = 0.0

    for tau in np.linspace(-tau_max, tau_max, PARK_NSTEPS):
        x_shift = shift_interp(x, fHz, tau)
        dS = np.mean(ref - x_shift)
        y_try = x_shift + dS
        err = np.mean((ref - y_try) ** 2)
        if err < best_err:
            best_err = err
            best_tau = float(tau)
            best_dS = float(dS)

    y = shift_interp(x, fHz, best_tau) + best_dS
    y = moving_average(y, PARK_SMOOTH_WIN)
    return y


def compensar_park(df, fcols, fHz, y_ref):
    X = df[fcols].to_numpy(float)
    Y = np.zeros_like(X)
    for i in range(len(X)):
        Y[i] = park_single(X[i], y_ref, fHz)
    df2 = df.copy()
    df2[fcols] = Y
    return df2


# ============================================================
# 10) RF CLEANUP PEQUENO
# ============================================================

def build_anchor_indices(n_points, n_anchors):
    n_anchors = int(min(max(3, n_anchors), n_points))
    idx = np.linspace(0, n_points - 1, n_anchors)
    return np.unique(np.round(idx).astype(int))


def interp_anchor_to_full(anchor_idx, y_anchor, n_points):
    grid = np.arange(n_points)
    Y = np.zeros((y_anchor.shape[0], n_points), dtype=float)
    for i in range(y_anchor.shape[0]):
        Y[i] = np.interp(grid, anchor_idx, y_anchor[i])
    return Y


def add_extra_features_matrix(X, T=None):
    X = np.asarray(X, dtype=float)
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, keepdims=True)
    amp = (X.max(axis=1) - X.min(axis=1)).reshape(-1, 1)
    q10 = np.quantile(X, 0.10, axis=1, keepdims=True)
    q50 = np.quantile(X, 0.50, axis=1, keepdims=True)
    q90 = np.quantile(X, 0.90, axis=1, keepdims=True)
    feats = [X, mu, sd, amp, q10, q50, q90]
    if T is not None:
        T = np.asarray(T, dtype=float).reshape(-1, 1)
        dT = np.abs(T - REF_TEMP)
        feats += [T, dT, np.sign(T - REF_TEMP)]
    return np.hstack(feats)


def train_rf_cleanup(df, fcols, fHz, healthy_by_temp, healthy_temps, y_ref, noise_floor):
    df_h = df[df["falha"] == 0].copy()
    Xh = df_h[fcols].to_numpy(float)
    Th = df_h["temperatura_c"].to_numpy(float)
    n_points = len(fcols)

    Y_base = []
    residuals = []
    for x, T in zip(Xh, Th):
        y_base, _ = signature_compensation_curve(x, T, fHz, healthy_by_temp, healthy_temps, y_ref, noise_floor)
        Y_base.append(y_base)
        residuals.append(y_ref - y_base)

    Y_base = np.asarray(Y_base, dtype=float)
    residuals = np.asarray(residuals, dtype=float)

    target_win = odd_window_from_frac(n_points, CLEANUP_TARGET_SMOOTH_WIN_FRAC, minimum=7)
    residual_smooth = moving_average_matrix(residuals, target_win)

    anchor_idx = build_anchor_indices(n_points, N_ANCHORS_CLEANUP)
    Y_anchor = residual_smooth[:, anchor_idx]

    X_input = moving_average_matrix(Y_base, target_win)
    X_in = add_extra_features_matrix(X_input, Th)

    rf = RandomForestRegressor(**RF_CLEANUP_PARAMS)
    rf.fit(X_in, Y_anchor)

    med = np.median(residual_smooth, axis=0)
    spread = np.percentile(np.abs(residual_smooth - med[None, :]), 90, axis=0)
    spread = np.maximum(spread, 1e-9)
    lower = med - CLEANUP_CLAMP_GAIN * spread
    upper = med + CLEANUP_CLAMP_GAIN * spread

    return {
        "rf": rf,
        "anchor_idx": anchor_idx,
        "lower": lower,
        "upper": upper,
    }


def predict_rf_cleanup(df_base_comp, fcols, rf_pack):
    X = df_base_comp[fcols].to_numpy(float)
    T = df_base_comp["temperatura_c"].to_numpy(float)
    n_points = X.shape[1]
    pred_win = odd_window_from_frac(n_points, CLEANUP_PRED_SMOOTH_WIN_FRAC, minimum=7)
    X_s = moving_average_matrix(X, pred_win)
    X_in = add_extra_features_matrix(X_s, T)

    pred_anchor = rf_pack["rf"].predict(X_in)
    pred_full = interp_anchor_to_full(rf_pack["anchor_idx"], pred_anchor, n_points)
    pred_full = moving_average_matrix(pred_full, pred_win)
    pred_full = np.clip(pred_full, rf_pack["lower"][None, :], rf_pack["upper"][None, :])
    return pred_full


def cleanup_peak_weights(y):
    win = odd_window_from_frac(len(y), CLEANUP_PEAK_PROTECT_SMOOTH_WIN_FRAC, minimum=5)
    p = peakness_score(y, win)
    w = 1.0 / (1.0 + CLEANUP_PEAK_PROTECT_GAIN * p)
    w = np.clip(w, CLEANUP_PEAK_PROTECT_MIN_WEIGHT, 1.0)
    return moving_average(w, win)


# ============================================================
# 11) COMPENSAR RF RESÍDUO V11 PEAK-ALIGNED CENTERED LESS-REF
# ============================================================

def compensar_rf_residuo_v11(df, fcols, fHz, healthy_by_temp, healthy_temps, y_ref):
    X = df[fcols].to_numpy(float)
    T_all = df["temperatura_c"].to_numpy(float)
    Y_base = np.zeros_like(X)
    info_rows = []

    noise_floor, noise_info = estimate_noise_floor_from_healthy(
        df, fcols, fHz, healthy_by_temp, healthy_temps, y_ref
    )
    noise_info.to_csv(os.path.join(PASTA_SAIDA, "info_shift_healthy_noise_v11.csv"), index=False)

    for i, (x, T) in enumerate(zip(X, T_all)):
        y_base, info = signature_compensation_curve(
            x, T, fHz, healthy_by_temp, healthy_temps, y_ref, noise_floor
        )
        Y_base[i] = y_base
        info_rows.append(info)

    df_base_comp = df.copy()
    df_base_comp[fcols] = Y_base

    Y_final = Y_base.copy()
    if USE_RF_CLEANUP:
        print("   Treinando RF cleanup pequeno na saída por assinatura (V11)...")
        rf_pack = train_rf_cleanup(df, fcols, fHz, healthy_by_temp, healthy_temps, y_ref, noise_floor)
        cleanup = predict_rf_cleanup(df_base_comp, fcols, rf_pack)

        for i in range(len(Y_final)):
            w_peak = cleanup_peak_weights(Y_base[i])
            dw = float(info_rows[i].get("damage_weight", 0.0))
            cleanup_blend_i = CLEANUP_BLEND * (1.0 - CLEANUP_DAMAGE_REDUCTION * dw)
            Y_final[i] = Y_base[i] + cleanup_blend_i * cleanup[i] * w_peak
            Y_final[i] = final_denoise_curve(Y_final[i])
    else:
        rf_pack = None

    df2 = df.copy()
    df2[fcols] = Y_final

    info_df = pd.DataFrame(info_rows)
    info_df.to_csv(os.path.join(PASTA_SAIDA, "info_rf_residuo_v11.csv"), index=False)

    return df2, {"rf_pack": rf_pack, "info": info_df, "noise_floor": noise_floor}


# ============================================================
# 12) MÉTRICAS
# ============================================================

def high_frequency_signature(x):
    win = odd_window_from_frac(len(x), HF_SIGNATURE_SMOOTH_WIN_FRAC, minimum=7)
    low = moving_average(x, win)
    return np.asarray(x, dtype=float) - low


def calcular_metricas(df_original, df_comp, fcols, y_ref, healthy_by_temp, healthy_temps, metodo, fmin_khz, fmax_khz):
    X_orig = df_original[fcols].to_numpy(float)
    X_comp = df_comp[fcols].to_numpy(float)

    rows = []
    for i in range(len(df_comp)):
        T = float(df_original.iloc[i]["temperatura_c"])
        h_T, _ = interpolate_healthy_curve(healthy_by_temp, healthy_temps, T)
        tau = estimate_healthy_shift(h_T, y_ref, np.arange(len(y_ref), dtype=float)) if False else 0.0

        y_orig = X_orig[i]
        y_comp = X_comp[i]

        # Assinatura esperada sem alinhamento fino para a métrica ficar conservadora.
        assinatura_esperada = y_orig - h_T
        assinatura_saida = y_comp - y_ref

        sig_orig = high_frequency_signature(y_orig)
        sig_comp = high_frequency_signature(y_comp)
        sig_corr = pearson_corr(sig_orig, sig_comp)

        delta = y_comp - y_orig

        rows.append({
            "metodo": metodo,
            "faixa_min_khz": float(fmin_khz),
            "faixa_max_khz": float(fmax_khz),
            "faixa_label": faixa_label(fmin_khz, fmax_khz),
            "temperatura_c": T,
            "falha": int(df_original.iloc[i]["falha"]),
            "RMSD": rmsd(y_comp, y_ref),
            "CCDM": ccdm(y_comp, y_ref),
            "DamageResidual_RMSD": rmsd(assinatura_saida, assinatura_esperada),
            "DamageResidual_CCDM": ccdm(assinatura_saida, assinatura_esperada),
            "assinatura_corr_original_comp": sig_corr,
            "delta_rms": float(np.sqrt(np.mean(delta ** 2))),
        })
    return pd.DataFrame(rows)


def resumir_metricas(df_metricas):
    return (
        df_metricas
        .groupby(["metodo", "faixa_min_khz", "faixa_max_khz", "faixa_label", "temperatura_c", "falha"], as_index=False)
        .agg(
            RMSD_medio=("RMSD", "mean"),
            RMSD_std=("RMSD", "std"),
            CCDM_medio=("CCDM", "mean"),
            CCDM_std=("CCDM", "std"),
            DamageResidual_RMSD_medio=("DamageResidual_RMSD", "mean"),
            DamageResidual_CCDM_medio=("DamageResidual_CCDM", "mean"),
            assinatura_corr_media=("assinatura_corr_original_comp", "mean"),
            delta_rms_medio=("delta_rms", "mean"),
            n_amostras=("RMSD", "size"),
        )
    )


def calcular_separacao(df_metricas):
    df_sum = resumir_metricas(df_metricas)
    rows = []
    group_cols = ["metodo", "faixa_min_khz", "faixa_max_khz", "faixa_label", "temperatura_c"]
    for keys, g in df_sum.groupby(group_cols):
        metodo, fmin, fmax, flabel, temp = keys
        gd = g.set_index("falha")
        if not all(d in gd.index for d in DANOS):
            continue
        for metrica in ["RMSD", "CCDM"]:
            v0 = float(gd.loc[0, f"{metrica}_medio"])
            v1 = float(gd.loc[1, f"{metrica}_medio"])
            v2 = float(gd.loc[2, f"{metrica}_medio"])
            rows.append({
                "metodo": metodo,
                "faixa_min_khz": fmin,
                "faixa_max_khz": fmax,
                "faixa_label": flabel,
                "temperatura_c": temp,
                "metrica": metrica,
                "D0": v0,
                "D1": v1,
                "D2": v2,
                "D1-D0": v1 - v0,
                "D2-D1": v2 - v1,
                "D2-D0": v2 - v0,
                "monotonico_D0_D1_D2": bool(v0 < v1 < v2),
            })
    return pd.DataFrame(rows)


# ============================================================
# 13) EXECUTAR UMA FAIXA
# ============================================================

def executar_uma_faixa(df_base, fmin_khz, fmax_khz):
    fcols, fHz = get_freq_columns(df_base, fmin_khz, fmax_khz)
    if len(fcols) < 5:
        raise ValueError(f"Poucas colunas na faixa {fmin_khz}-{fmax_khz} kHz.")

    df_use = df_base[["temperatura_c", "falha"] + fcols].copy()
    print(f"\n🔹 Faixa {fmin_khz:.1f}–{fmax_khz:.1f} kHz | {len(fcols)} pontos")
    t0 = time.time()

    healthy_by_temp, healthy_temps, y_ref, ref_temp_used = get_healthy_references_by_temperature(df_use, fcols)

    metricas = []
    curvas_por_metodo = {"Original": df_use.copy()}

    if RUN_ORIGINAL:
        metricas.append(calcular_metricas(df_use, df_use, fcols, y_ref, healthy_by_temp, healthy_temps, "Original", fmin_khz, fmax_khz))

    if RUN_PARK:
        print("   Aplicando Park...")
        df_park = compensar_park(df_use, fcols, fHz, y_ref)
        curvas_por_metodo["Park"] = df_park
        metricas.append(calcular_metricas(df_use, df_park, fcols, y_ref, healthy_by_temp, healthy_temps, "Park", fmin_khz, fmax_khz))

    info_rf = None
    if RUN_RF_RESIDUO_V11:
        print("   Aplicando RF resíduo V11...")
        df_rf, info_rf = compensar_rf_residuo_v11(df_use, fcols, fHz, healthy_by_temp, healthy_temps, y_ref)
        curvas_por_metodo["RF_residuo_v11_peakaligned_centered_lessref"] = df_rf
        metricas.append(calcular_metricas(df_use, df_rf, fcols, y_ref, healthy_by_temp, healthy_temps, "RF_residuo_v11_peakaligned_centered_lessref", fmin_khz, fmax_khz))

    print(f"✅ Faixa concluída em {time.time() - t0:.1f} s")

    return {
        "fcols": fcols,
        "fHz": fHz,
        "df_use": df_use,
        "healthy_by_temp": healthy_by_temp,
        "healthy_temps": healthy_temps,
        "y_ref": y_ref,
        "curvas_por_metodo": curvas_por_metodo,
        "metricas": pd.concat(metricas, ignore_index=True),
        "info_rf": info_rf,
    }


# ============================================================
# 14) GRÁFICOS
# ============================================================

def plot_curvas_exemplo(resultado_faixa, temps_alvo=TEMPERATURAS_CURVAS):
    df_use = resultado_faixa["df_use"]
    fcols = resultado_faixa["fcols"]
    fHz = resultado_faixa["fHz"]
    y_ref = resultado_faixa["y_ref"]
    curvas_por_metodo = resultado_faixa["curvas_por_metodo"]

    fmin = extract_freq_hz(fcols[0]) / 1e3
    fmax = extract_freq_hz(fcols[-1]) / 1e3
    fkhz = fHz / 1e3

    temps_usadas, mapa = escolher_temperaturas_validas(df_use, temps_alvo, danos=DANOS)
    mapa.to_csv(os.path.join(PASTA_CURVAS, f"temperaturas_usadas_curvas_{fmin:.1f}_{fmax:.1f}.csv"), index=False)

    for T in temps_usadas:
        fig, axes = plt.subplots(1, len(DANOS), figsize=(7.4 * len(DANOS), 6.2), dpi=300, sharey=True)
        if len(DANOS) == 1:
            axes = [axes]

        for ax, dano in zip(axes, DANOS):
            mask = np.isclose(df_use["temperatura_c"], T) & (df_use["falha"] == dano)
            idxs = np.where(mask.to_numpy())[0]
            if len(idxs) == 0:
                ax.set_title(f"Dano {dano}\nsem amostra")
                continue
            idx = int(idxs[0])

            ax.plot(fkhz, y_ref, "--", c="black", lw=1.8, label=f"Referência {REF_TEMP:.0f}°C")

            y_orig = df_use.iloc[idx][fcols].to_numpy(float)
            ax.plot(fkhz, y_orig, c="tab:red", alpha=0.36, lw=1.35, label=f"Original {format_temp(T)}°C")

            for metodo in ["Park", "RF_residuo_v11_peakaligned_centered_lessref"]:
                if metodo not in curvas_por_metodo:
                    continue
                y = curvas_por_metodo[metodo].iloc[idx][fcols].to_numpy(float)
                ax.plot(
                    fkhz,
                    y,
                    c=CORES_METODO.get(metodo, None),
                    lw=2.35,
                    alpha=0.95,
                    label=NOMES_METODO.get(metodo, metodo),
                )

            ax.set_title(f"Dano {dano}", fontsize=25, pad=12)
            ax.set_xlabel("Frequência (kHz)", fontsize=22)
            estilo_eixos(ax)

        axes[0].set_ylabel("Impedância", fontsize=23)
        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="lower center", ncol=4, frameon=True, fontsize=13, bbox_to_anchor=(0.5, -0.04))
        fig.suptitle(f"Curvas exemplo — faixa {fmin:.1f}–{fmax:.1f} kHz — T = {format_temp(T)}°C", fontsize=27, y=1.02)
        fig.tight_layout(rect=[0, 0.08, 1, 0.95])
        nome = f"curvas_exemplo_Park_vs_RFV11_{fmin:.1f}_{fmax:.1f}kHz_T{format_temp(T).replace('-', 'm')}C"
        salvar_fig(fig, PASTA_CURVAS, nome)
        plt.show()


def preparar_resumo_temperatura(df_metricas, fmin, fmax):
    sub = df_metricas[
        np.isclose(df_metricas["faixa_min_khz"], float(fmin)) &
        np.isclose(df_metricas["faixa_max_khz"], float(fmax))
    ].copy()
    temps_usadas, mapa = escolher_temperaturas_validas(sub, TEMPERATURAS_ALVO_PLOT, danos=DANOS)
    sub = sub[sub["temperatura_c"].isin(temps_usadas)].copy()
    return resumir_metricas(sub), temps_usadas, mapa


def plot_metricas_por_temperatura(df_metricas, fmin, fmax):
    df_sum, temps_usadas, mapa = preparar_resumo_temperatura(df_metricas, fmin, fmax)
    mapa.to_csv(os.path.join(PASTA_METRICAS, f"temperaturas_usadas_{fmin:.1f}_{fmax:.1f}kHz.csv"), index=False)

    for metodo in [m for m in ["Park", "RF_residuo_v11_peakaligned_centered_lessref"] if m in df_sum["metodo"].unique()]:
        fig, axes = plt.subplots(1, 2, figsize=(18, 7.2), dpi=300)
        for ax, metrica in zip(axes, ["RMSD", "CCDM"]):
            col_media = f"{metrica}_medio"
            col_std = f"{metrica}_std"
            for dano in DANOS:
                sd = df_sum[(df_sum["metodo"] == metodo) & (df_sum["falha"] == dano)].copy()
                sd = sd.sort_values("temperatura_c").set_index("temperatura_c").reindex(temps_usadas)
                ax.errorbar(
                    temps_usadas,
                    sd[col_media].to_numpy(float),
                    yerr=sd[col_std].fillna(0.0).to_numpy(float),
                    marker="o",
                    linewidth=2.5,
                    markersize=7,
                    capsize=4,
                    label=f"Dano {dano}",
                    color=CORES_DANO.get(dano, None),
                )
            ax.axvline(REF_TEMP, color="black", linestyle="--", linewidth=1.1, alpha=0.75)
            ax.set_xlabel("Temperatura (°C)", fontsize=22)
            ax.set_ylabel(metrica, fontsize=22)
            ax.set_title(metrica, fontsize=24, pad=12)
            ax.set_xticks(temps_usadas)
            ax.set_xticklabels([format_temp(t) for t in temps_usadas])
            estilo_eixos(ax)
        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="lower center", ncol=3, frameon=True, fontsize=14, bbox_to_anchor=(0.5, -0.06))
        fig.suptitle(f"{NOMES_METODO.get(metodo, metodo)} — RMSD e CCDM por temperatura — faixa {faixa_label(fmin, fmax)}", fontsize=26, y=1.02)
        fig.tight_layout(rect=[0, 0.08, 1, 0.95])
        nome = f"metricas_temperatura_{metodo}_{fmin:.1f}_{fmax:.1f}kHz"
        salvar_fig(fig, PASTA_METRICAS, nome)
        plt.show()


def plot_comparacao_geral_metodos(df_metricas, fmin, fmax):
    sub = df_metricas[
        np.isclose(df_metricas["faixa_min_khz"], float(fmin)) &
        np.isclose(df_metricas["faixa_max_khz"], float(fmax)) &
        (df_metricas["metodo"].isin(["Original", "Park", "RF_residuo_v11_peakaligned_centered_lessref"]))
    ].copy()
    resumo = sub.groupby(["metodo", "falha"], as_index=False).agg(
        RMSD=("RMSD", "mean"),
        CCDM=("CCDM", "mean"),
        DamageResidual_RMSD=("DamageResidual_RMSD", "mean"),
    )
    metodos = [m for m in ["Original", "Park", "RF_residuo_v11_peakaligned_centered_lessref"] if m in resumo["metodo"].unique()]
    danos = sorted(resumo["falha"].unique())

    fig, axes = plt.subplots(1, 2, figsize=(18, 7.2), dpi=300)
    width = 0.22
    x = np.arange(len(metodos))

    for ax, metrica in zip(axes, ["RMSD", "CCDM"]):
        for i, dano in enumerate(danos):
            vals = []
            for metodo in metodos:
                v = resumo.loc[(resumo["metodo"] == metodo) & (resumo["falha"] == dano), metrica]
                vals.append(float(v.iloc[0]) if len(v) else np.nan)
            ax.bar(x + (i - 1) * width, vals, width=width, edgecolor="black", linewidth=0.8, label=f"Dano {dano}", color=CORES_DANO.get(dano, None), alpha=0.85)
        ax.set_xticks(x)
        ax.set_xticklabels([NOMES_METODO.get(m, m) for m in metodos], rotation=12, ha="right")
        ax.set_ylabel(f"{metrica} médio")
        ax.set_title(metrica)
        estilo_eixos(ax)
    axes[1].legend(frameon=True, loc="best")
    fig.suptitle(f"Comparação geral — faixa {faixa_label(fmin, fmax)}", fontsize=26, y=1.02)
    fig.tight_layout()
    salvar_fig(fig, PASTA_METRICAS, f"comparacao_geral_Park_vs_RFV11_{fmin:.1f}_{fmax:.1f}kHz")
    plt.show()


def plot_separacao_por_temperatura(df_metricas, fmin, fmax):
    sep = calcular_separacao(df_metricas)
    sep = sep[
        np.isclose(sep["faixa_min_khz"], float(fmin)) &
        np.isclose(sep["faixa_max_khz"], float(fmax))
    ].copy()
    path = os.path.join(PASTA_METRICAS, f"separacao_danos_{fmin:.1f}_{fmax:.1f}kHz.csv")
    sep.to_csv(path, index=False)
    print(f"✅ Separação salva em: {path}")

    for metodo in [m for m in ["Park", "RF_residuo_v11_peakaligned_centered_lessref"] if m in sep["metodo"].unique()]:
        fig, axes = plt.subplots(1, 2, figsize=(18, 7.2), dpi=300)
        for ax, metrica in zip(axes, ["RMSD", "CCDM"]):
            sm = sep[(sep["metodo"] == metodo) & (sep["metrica"] == metrica)].sort_values("temperatura_c")
            for col, label, marker in [
                ("D1-D0", "Dano 1 − Dano 0", "o"),
                ("D2-D1", "Dano 2 − Dano 1", "s"),
                ("D2-D0", "Dano 2 − Dano 0", "^"),
            ]:
                ax.plot(sm["temperatura_c"], sm[col], marker=marker, linewidth=2.5, markersize=7, label=label)
            ax.axhline(0, color="black", linewidth=1.2)
            ax.axvline(REF_TEMP, color="black", linestyle="--", linewidth=1.1, alpha=0.75)
            ax.set_xlabel("Temperatura (°C)", fontsize=22)
            ax.set_ylabel(f"Separação em {metrica}", fontsize=22)
            ax.set_title(metrica, fontsize=24, pad=12)
            estilo_eixos(ax)
        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="lower center", ncol=3, frameon=True, fontsize=13, bbox_to_anchor=(0.5, -0.06))
        fig.suptitle(f"{NOMES_METODO.get(metodo, metodo)} — separação entre danos — faixa {faixa_label(fmin, fmax)}", fontsize=26, y=1.02)
        fig.tight_layout(rect=[0, 0.08, 1, 0.95])
        nome = f"separacao_temperatura_{metodo}_{fmin:.1f}_{fmax:.1f}kHz"
        salvar_fig(fig, PASTA_METRICAS, nome)
        plt.show()


# ============================================================
# 15) MAIN
# ============================================================

def rodar_tudo_limpo():
    print("=" * 90)
    print("RF RESÍDUO V11 PEAK-ALIGNED CENTERED LESS-REF — ASSINATURA PRESERVADA + PARK")
    print("=" * 90)

    if not os.path.exists(ARQ_BASE):
        raise FileNotFoundError(f"Não encontrei {ARQ_BASE}. Coloque o arquivo na pasta do script.")

    print("🔹 Carregando base...")
    df_base = pd.read_pickle(ARQ_BASE).reset_index(drop=True)

    required = ["temperatura_c", "falha"]
    missing = [c for c in required if c not in df_base.columns]
    if missing:
        raise ValueError(f"A base não tem as colunas obrigatórias: {missing}")

    df_base["temperatura_c"] = pd.to_numeric(df_base["temperatura_c"], errors="coerce")
    df_base["falha"] = pd.to_numeric(df_base["falha"], errors="coerce").astype(int)

    todas_metricas = []
    resultados_por_faixa = {}

    for fmin, fmax in FAIXAS_ANALISE:
        res = executar_uma_faixa(df_base, fmin, fmax)
        resultados_por_faixa[(float(fmin), float(fmax))] = res
        todas_metricas.append(res["metricas"])

    df_metricas = pd.concat(todas_metricas, ignore_index=True)
    df_resumo = resumir_metricas(df_metricas)
    df_sep = calcular_separacao(df_metricas)

    path_metricas = os.path.join(PASTA_SAIDA, "metricas_amostra_a_amostra.csv")
    path_resumo = os.path.join(PASTA_SAIDA, "resumo_metricas.csv")
    path_sep = os.path.join(PASTA_SAIDA, "separacao_danos.csv")

    df_metricas.to_csv(path_metricas, index=False)
    df_resumo.to_csv(path_resumo, index=False)
    df_sep.to_csv(path_sep, index=False)

    print("\n✅ CSVs salvos:")
    print(path_metricas)
    print(path_resumo)
    print(path_sep)

    print("\n📊 Gerando gráficos importantes...")
    for fmin, fmax in FAIXAS_ANALISE:
        res = resultados_por_faixa[(float(fmin), float(fmax))]
        plot_curvas_exemplo(res, temps_alvo=TEMPERATURAS_CURVAS)
        plot_metricas_por_temperatura(df_metricas, fmin, fmax)
        plot_separacao_por_temperatura(df_metricas, fmin, fmax)
        plot_comparacao_geral_metodos(df_metricas, fmin, fmax)

    print("\nResumo médio por método/dano:")
    print(
        df_metricas
        .groupby(["metodo", "falha"])[["RMSD", "CCDM", "DamageResidual_RMSD", "delta_rms"]]
        .mean()
        .round(6)
    )

    print("\n" + "=" * 90)
    print("✅ FINALIZADO")
    print(f"📁 Pasta de saída: {PASTA_SAIDA}")
    print("=" * 90)

    return df_metricas, df_resumo, df_sep, resultados_por_faixa


if __name__ == "__main__":
    df_metricas_v11, df_resumo_v11, df_sep_v11, resultados_v11 = rodar_tudo_limpo()


In [ ]:
# -*- coding: utf-8 -*-
"""
GRÁFICO DE BARRAS — RMSD/CCDM POR TEMPERATURA — RF
===================================================

Use este arquivo depois de rodar o código do RF.
Ele lê o CSV metricas_amostra_a_amostra.csv e gera um gráfico no estilo:

    - painel esquerdo: RMSD
    - painel direito: CCDM
    - eixo x: temperatura
    - barras agrupadas: Original, Park e RF, separados por dano 0/1/2

Como usar:
----------
1) Rode primeiro o código principal do RF.
2) Coloque este arquivo na mesma pasta do notebook/script.
3) Rode este arquivo.

Se o caminho do CSV for diferente, altere ARQUIVO_METRICAS abaixo.
"""

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ============================================================
# CONFIGURAÇÕES
# ============================================================

PASTA_SAIDA_RF = "resultados_rf_residuo_v11_peakaligned_centered_lessref_30_40_vs_park"
ARQUIVO_METRICAS = os.path.join(PASTA_SAIDA_RF, "metricas_amostra_a_amostra.csv")

PASTA_GRAFICO = os.path.join(PASTA_SAIDA_RF, "graficos", "barras_metricas_por_temperatura")
os.makedirs(PASTA_GRAFICO, exist_ok=True)

NOME_METODO_RF = "RF_residuo_v11_peakaligned_centered_lessref"
METODOS_ORDEM = ["Original", "Park", NOME_METODO_RF]
DANOS_ORDEM = [0, 1, 2]

# Para não poluir o gráfico, escolha cerca de 6 temperaturas.
# Se alguma temperatura não existir exatamente no CSV, o código usa a mais próxima disponível.
# Altere esta lista se quiser outros pontos.
TEMPERATURAS_PLOT = [-10, 10, 30, 50, 70, 80]

SALVAR_PDF = True
MOSTRAR_GRAFICO = True

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 17,
    "axes.labelsize": 22,
    "axes.titlesize": 24,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 13,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# Cores: cada dano mantém a família de cor; cada método muda a intensidade.
CORES = {
    ("Original", 0): "#9ecae1",   # azul claro
    ("Original", 1): "#fdd0a2",   # laranja claro
    ("Original", 2): "#f4a6a6",   # vermelho claro

    ("Park", 0): "#6baed6",
    ("Park", 1): "#fdae6b",
    ("Park", 2): "#e76f6f",

    (NOME_METODO_RF, 0): "#1f77b4",
    (NOME_METODO_RF, 1): "#ff7f0e",
    (NOME_METODO_RF, 2): "#d62728",
}

LABEL_METODO = {
    "Original": "Original",
    "Park": "Park",
    NOME_METODO_RF: "RF resíduo V11",
}

# ============================================================
# FUNÇÕES
# ============================================================

def procurar_csv_fallback():
    """Procura o CSV mais recente caso o caminho configurado não exista."""
    candidatos = glob.glob("**/metricas_amostra_a_amostra.csv", recursive=True)
    candidatos = [c for c in candidatos if os.path.isfile(c)]
    if not candidatos:
        return None
    candidatos = sorted(candidatos, key=os.path.getmtime, reverse=True)
    return candidatos[0]


def carregar_metricas():
    path = ARQUIVO_METRICAS
    if not os.path.exists(path):
        alt = procurar_csv_fallback()
        if alt is None:
            raise FileNotFoundError(
                "Não encontrei metricas_amostra_a_amostra.csv. "
                "Rode primeiro o código principal do RF ou ajuste ARQUIVO_METRICAS."
            )
        print(f"⚠️ Não achei o caminho padrão. Usando CSV encontrado: {alt}")
        path = alt

    df = pd.read_csv(path)

    # Normaliza nome da coluna de método se necessário.
    if "metodo" not in df.columns and "Metodo" in df.columns:
        df = df.rename(columns={"Metodo": "metodo"})

    obrig = {"metodo", "temperatura_c", "falha", "RMSD", "CCDM"}
    faltando = obrig - set(df.columns)
    if faltando:
        raise ValueError(f"CSV sem colunas obrigatórias: {faltando}")

    df["temperatura_c"] = pd.to_numeric(df["temperatura_c"], errors="coerce")
    df["falha"] = pd.to_numeric(df["falha"], errors="coerce").astype(int)
    return df, path


def estilo_eixos(ax):
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def format_temp(T):
    T = float(T)
    return str(int(T)) if T.is_integer() else f"{T:.1f}"


def selecionar_temperaturas_plot(temps_disponiveis, temps_desejadas=TEMPERATURAS_PLOT):
    """Seleciona poucas temperaturas para o gráfico, pegando a mais próxima disponível."""
    temps_disponiveis = np.array(sorted(pd.Series(temps_disponiveis).dropna().unique()), dtype=float)
    if len(temps_disponiveis) == 0:
        return temps_disponiveis

    selecionadas = []
    for T in temps_desejadas:
        idx = int(np.argmin(np.abs(temps_disponiveis - float(T))))
        T_real = float(temps_disponiveis[idx])
        if not any(np.isclose(T_real, t) for t in selecionadas):
            selecionadas.append(T_real)

    # Se por algum motivo ficaram menos de 6, completa com temperaturas espaçadas.
    alvo_n = min(6, len(temps_disponiveis))
    if len(selecionadas) < alvo_n:
        idxs = np.linspace(0, len(temps_disponiveis) - 1, alvo_n).round().astype(int)
        for T_real in temps_disponiveis[idxs]:
            T_real = float(T_real)
            if not any(np.isclose(T_real, t) for t in selecionadas):
                selecionadas.append(T_real)

    return np.array(sorted(selecionadas), dtype=float)


def preparar_resumo(df):
    metodos_presentes = [m for m in METODOS_ORDEM if m in df["metodo"].unique()]
    if len(metodos_presentes) == 0:
        raise ValueError(
            "Nenhum dos métodos esperados foi encontrado no CSV. "
            f"Métodos no CSV: {sorted(df['metodo'].dropna().unique())}"
        )

    temps = selecionar_temperaturas_plot(df["temperatura_c"].dropna().unique())
    danos_presentes = [d for d in DANOS_ORDEM if d in set(df["falha"].unique())]

    resumo = (
        df[df["metodo"].isin(metodos_presentes) & df["falha"].isin(danos_presentes)]
        .groupby(["metodo", "temperatura_c", "falha"], as_index=False)
        .agg(
            RMSD=("RMSD", "mean"),
            CCDM=("CCDM", "mean"),
            n=("RMSD", "size"),
        )
    )
    return resumo, temps, metodos_presentes, danos_presentes


def plot_barras_rmsd_ccdm_por_temperatura(df):
    resumo, temps, metodos, danos = preparar_resumo(df)

    n_met = len(metodos)
    n_dan = len(danos)
    n_barras = n_met * n_dan

    x = np.arange(len(temps), dtype=float)
    largura_total = 0.82
    largura_barra = largura_total / max(n_barras, 1)

    fig, axes = plt.subplots(1, 2, figsize=(17, 8.2), dpi=300)

    for ax, metrica in zip(axes, ["RMSD", "CCDM"]):
        k = 0
        for metodo in metodos:
            for dano in danos:
                vals = []
                for T in temps:
                    v = resumo.loc[
                        (resumo["metodo"] == metodo)
                        & np.isclose(resumo["temperatura_c"], T)
                        & (resumo["falha"] == dano),
                        metrica,
                    ]
                    vals.append(float(v.iloc[0]) if len(v) else np.nan)

                desloc = -largura_total / 2 + (k + 0.5) * largura_barra
                ax.bar(
                    x + desloc,
                    vals,
                    width=largura_barra * 0.92,
                    color=CORES.get((metodo, dano), None),
                    edgecolor="black",
                    linewidth=0.35,
                    alpha=0.95,
                )
                k += 1

        ax.set_xlabel("Temperatura (°C)")
        ax.set_ylabel(metrica)
        ax.set_title(metrica)
        ax.set_xticks(x)
        ax.set_xticklabels([format_temp(T) for T in temps])
        estilo_eixos(ax)

    handles = []
    for metodo in metodos:
        for dano in danos:
            handles.append(
                Patch(
                    facecolor=CORES.get((metodo, dano), "gray"),
                    edgecolor="black",
                    linewidth=0.35,
                    label=f"{LABEL_METODO.get(metodo, metodo)} — Dano {dano}",
                )
            )

    fig.legend(
        handles=handles,
        loc="upper center",
        ncol=3,
        frameon=False,
        bbox_to_anchor=(0.5, 1.05),
    )

    fig.suptitle("Comparação RMSD/CCDM por temperatura — RF", fontsize=26, y=1.13)
    fig.tight_layout(rect=[0, 0, 1, 0.93])

    nome = "barras_RMSD_CCDM_6_temperaturas_RF"
    png = os.path.join(PASTA_GRAFICO, nome + ".png")
    fig.savefig(png, dpi=600, bbox_inches="tight", facecolor="white")
    print(f"✅ PNG salvo: {png}")

    if SALVAR_PDF:
        pdf = os.path.join(PASTA_GRAFICO, nome + ".pdf")
        fig.savefig(pdf, bbox_inches="tight", facecolor="white")
        print(f"✅ PDF salvo: {pdf}")

    if MOSTRAR_GRAFICO:
        plt.show()
    else:
        plt.close(fig)

    return fig


# ============================================================
# RODAR
# ============================================================

if __name__ == "__main__":
    df_metricas, path_usado = carregar_metricas()
    print(f"📄 CSV usado: {path_usado}")
    plot_barras_rmsd_ccdm_por_temperatura(df_metricas)
